# Student Task: Build a RAG Question-Answering System

## Goal
In this task, you will build a Retrieval-Augmented Generation (RAG) pipeline over the provided `RAG and Fine-tuning - Final.pdf` file.

You will implement these stages:

```text
PDF -> Text -> Chunks -> Embeddings -> Retrieval -> Augmented Prompt -> Grounded Answer
```

Complete every section marked `TODO`.

## Learning objectives

By the end of this task, you should be able to:

- Explain why RAG is useful for private, changing, or large information sources.
- Load and prepare a PDF for semantic search.
- Split a document into overlapping chunks.
- Create embeddings and retrieve the most relevant chunks with cosine similarity.
- Augment a prompt with retrieved context.
- Generate an answer that is grounded in the source document.
- Compare RAG with fine-tuning and describe the purpose of LoRA/QLoRA.

## Task requirements

Your final notebook should:

1. Read the supplied PDF with `pypdf`.
2. Create chunks with a configurable size and overlap.
3. Embed all chunks and the user's question with OpenAI.
4. Retrieve the top 3 relevant chunks using cosine similarity.
5. Generate a concise answer using only the retrieved context.
6. Print the retrieved sources, answer, and a simple grounding check.
7. Answer the conceptual questions near the end.

Do not place your API key directly in the notebook.

## 1. Setup

In [1]:
%pip install -q -U openai pypdf numpy python-dotenv

import os
from pathlib import Path
import re
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader

load_dotenv()
api_key = os.getenv("OPENROUTER_API_KEY")
client = OpenAI(api_key=api_key, base_url="https://openrouter.ai/api/v1")
CHAT_MODEL = "openai/gpt-5.6-sol"
EMBEDDING_MODEL = "openai/text-embedding-3-small"
PDF_PATH = Path("RAG and Fine-tuning - Final.pdf")

print("Setup complete")


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Setup complete


## 2. Load the source document

In [2]:
# TODO 1: Read every page from PDF_PATH and combine the extracted text.
# Save the result in document_text and print its character count.
reader = PdfReader(PDF_PATH)
pages_text = []
for page in reader.pages:
    pages_text.append(page.extract_text())
document_text = "\n".join(pages_text)

print("Document characters:", len(document_text) if document_text else 0)

Document characters: 38671


## 3. Chunk the document

Chunking makes retrieval possible because the system can search meaningful sections instead of sending the entire PDF to the model. Keep the overlap so ideas split across boundaries are less likely to be lost.

In [3]:
CHUNK_SIZE = 900
CHUNK_OVERLAP = 150

# TODO 2: Write chunk_text(text, chunk_size, overlap).
# Return a list of non-empty overlapping text chunks.
def chunk_text(text: str, chunk_size: int = 900, overlap: int = 150) -> list:
    step = chunk_size - overlap
    chunks = []
    for start in range(0, len(text), step):
        chunk = text[start:start + chunk_size].strip()
        if chunk:
            chunks.append(chunk)

    return chunks

chunks = chunk_text(document_text, CHUNK_SIZE, CHUNK_OVERLAP)
print("Number of chunks:", len(chunks))
print("First chunk preview:\n", chunks[0][:500])

Number of chunks: 52
First chunk preview:
 arXiv:2505.10792v2  [cs.CL]  19 May 2025
Finetune-RAG: Fine-Tuning Language Models to
Resist Hallucination in Retrieval-Augmented
Generation
Zhan Peng Lee
Pints AI Labs
zhanpeng.lee@pints.co
Andre Lin∗
Pints AI Labs
andrelim444@gmail.com
Calvin Tan
Pints AI Labs
calvin@pints.co
Abstract
Retrieval-Augmented Generation (RAG) has emerged as a powerful framework to
improve factuality in large language models (LLMs) by grounding their outputs in
retrieved documents. However, ensuring perfect retrieva


## 4. Create embeddings and an in-memory vector index

The embedding represents meaning as a vector. The index below is intentionally simple: it stores vectors in NumPy and uses cosine similarity.

In [4]:
# TODO 3: Complete embed_texts so it returns one list of embedding vectors per input text.
def embed_texts(texts):
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return [item.embedding for item in response.data]

# TODO 4: Complete cosine_similarity_matrix.
# It should return the cosine similarity between one query vector and every row in matrix.
def cosine_similarity_matrix(query_vector: np.ndarray, matrix: np.ndarray) -> np.ndarray:
    similarities = []
    for row in matrix:
        similarity = np.dot(query_vector, row) / (np.linalg.norm(query_vector) * np.linalg.norm(row))
        similarities.append(similarity)

    return np.array(similarities)

chunk_embeddings = np.array(embed_texts(chunks), dtype=np.float32)
print("Embedding matrix shape:", chunk_embeddings.shape)

Embedding matrix shape: (52, 1536)


## 5. Retrieve relevant context

In [5]:
question = "What is Finetune-RAG and how does it reduce hallucination in RAG systems?"
TOP_K = 3

# TODO 5: Embed the question, calculate similarities, and select the top TOP_K chunks.
# Save the selected text in retrieved_chunks and their scores in retrieved_scores.
query_embedding = embed_texts([question])[0]
similarities = cosine_similarity_matrix(np.array(query_embedding, dtype=np.float32), chunk_embeddings)
top_indices = np.argsort(similarities)[::-1][:TOP_K]

retrieved_chunks = []
retrieved_scores = []
for index in top_indices:
    retrieved_chunks.append(chunks[index])
    retrieved_scores.append(similarities[index])

for rank, (score, chunk) in enumerate(zip(retrieved_scores, retrieved_chunks), start=1):
    print(f"[{rank}] score={score:.3f}\n{chunk[:400]}\n")

[1] score=0.743
l.
8 Conclusion
In this work, we present Finetune-RAG, a simple yet effective method for reducing hallucination in
Retrieval-Augmented Generation (RAG) through supervised fine-tuning. Rather than focusing on
retrieval quality, Finetune-RAG trains the generation model to rely solely on factual context while
ignoring misleading information, with no architectural changes required.
We constructed a di

[2] score=0.730
ned retrieval mechanisms such as reranker-
aware retrievers or contrastively trained retrievers could lead to further improvements in
factual accuracy and context filtering.
• Multimodal extensions : Hallucination is not limited to text-based models. Ex-
tending Finetune-RAG to multimodal settings, such as image-caption retrieval or
code+documentation generation, may help build more robust grounde

[3] score=0.664
arXiv:2505.10792v2  [cs.CL]  19 May 2025
Finetune-RAG: Fine-Tuning Language Models to
Resist Hallucination in Retrieval-Augmented
Generation
Zhan Pe

## 6. Augment the prompt and generate a grounded answer

The model must use the retrieved context and must say when the answer is not supported by the source.

In [6]:
# TODO 6: Build context from retrieved_chunks.
context = "\n\n".join(retrieved_chunks)

# TODO 7: Write an augmented prompt containing the question and context.
augmented_prompt = f"""Answer the question using only the context provided below. Give a concise answer.
If the context does not contain the answer, say that the document does not provide enough information.

Context:
{context}

Question:
{question}"""

# TODO 8: Call the Responses API and save the answer in answer.
response = client.responses.create(model=CHAT_MODEL, input=augmented_prompt)
answer = response.output_text

print("Answer:")
print(answer)

Answer:
Finetune-RAG is a supervised fine-tuning method for Retrieval-Augmented Generation systems. It reduces hallucination by training the generation model to rely only on factual retrieved context and ignore irrelevant or misleading information, without requiring architectural changes or perfect retrieval.


## 7. Evaluate grounding

This is a small educational check, not a complete evaluation system. Inspect whether important terms from the retrieved context appear in the answer and whether the answer admits missing evidence.

In [7]:
# TODO 9: Normalize text and calculate how many distinctive context terms appear in the answer.
def normalize_words(text):
    return set(re.findall(r"[a-zA-Z]{4,}", text.lower()))

context_terms = normalize_words(context or "")
answer_terms = normalize_words(answer or "")
overlap = context_terms & answer_terms
grounding_ratio = len(overlap) / max(len(answer_terms), 1)

print(f"Distinctive-term overlap: {len(overlap)}")
print(f"Simple grounding ratio: {grounding_ratio:.2%}")
print("Retrieved context used:", bool(context))

Distinctive-term overlap: 22
Simple grounding ratio: 81.48%
Retrieved context used: True


## Summary of Conceptual Reflection Answers

1. **What do indexing, chunking, retrieval, and augmentation each do in a RAG workflow?**
   Chunking splits the document into overlapping searchable pieces. Indexing embeds and stores those chunks. Retrieval finds the chunks most similar to the question. Augmentation inserts those chunks into the prompt so the model answers from them.

2. **Why can poor retrieval produce a poor answer even when the language model is capable?**
   The model can only work with what's in the prompt. Bad retrieval forces it to either guess from its own training knowledge or admit the answer isn't supported, no matter how capable the model is.

3. **How is a vector database different from a traditional keyword database?**
   Keyword databases match literal words or substrings. Vector databases match by meaning through embedding similarity, so they find relevant content even when the wording differs completely.

4. **When is RAG a better choice than fine-tuning? When might fine-tuning be better?**
   RAG is better for information that's private, large, or changes frequently, since updating knowledge just means updating the indexed documents. Fine-tuning is better for changing the model's behavior, tone, or skills rather than its facts.

5. **What do LoRA and QLoRA change during fine-tuning, and why are base model weights often frozen?**
   Both freeze the base model's original weights and train small added low-rank matrices instead, updating far fewer parameters. QLoRA also quantizes the base model to cut memory use. Weights stay frozen to keep training cheap and to avoid overwriting the model's existing knowledge.

## Submission checklist

- [ ] All `TODO` sections are completed.
- [ ] The notebook runs from top to bottom without errors.
- [ ] The PDF is loaded and split into overlapping chunks.
- [ ] The top retrieved chunks and similarity scores are printed.
- [ ] The answer is generated from retrieved context.
- [ ] The grounding check is printed and interpreted.
- [ ] The conceptual reflection questions are answered.
- [ ] No API key is written directly in the notebook.